# Helpers


## ¿Qué son los helpers?
En Bash, los helpers no son algo “oficial” del lenguaje, sino un concepto práctico.

> Un helper es simplemente una **función reutilizable** que te ayuda a no repetir código.

In [2]:
%%bash
# Uso sin helpers
echo "Iniciando proceso ..."
date
echo "Proceso terminado"


Iniciando proceso ...
Mon Apr 27 20:52:22 UTC 2026
Proceso terminado


In [3]:
%%bash
# Uso con helpers
log() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1"
}

log "Iniciando proceso ..."
log "Proceso terminado ..."

[2026-04-27 20:57:42] Iniciando proceso ...
[2026-04-27 20:57:42] Proceso terminado ...


Aquí `log` es un helper porque:
  * Encapsula lógica común
  * Se puede reutilizar
  * Hace el código más limpio

### Helpers en archivos separados
Cuando tus scripts crecen, separas helpers en otro archivo.

#### `helpers.sh`
```bash
log() {
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1"
}

error() {
  echo "ERROR: $1" >&2
}
```

#### `main.sh`
```bash
source helpers.sh

log "Script iniciado"
error "Algo salió mal"
```

> `source` (o `.`) importa funciones de otro archivo.

### Problema real: inclusión múltiple
Aquí aparece algo importante.

Si haces esto varias veces:
```bash
source helpers.sh
source helpers.sh
```
Bash volverá a cargar todo:
  * redefinir funciones
  * ejecutar código global
  * posibles efectos secundarios

En scripts pequeños no importa ... en proyectos grandes sí y ahí entra `guard de inclusión`

## Guard de inclusión
Es una técnica para evitar que un archivo se cargue más de una vez.

Es como decir:

> "Si ya cargué este archivo, no lo vuelvas a ejecutar"

### Guard de inclusión básico
Se usa una variable como bandera:

```bash
# guard de inclusión
if [[ -n "$HELPERS_LOADED" ]]; then
  return
fi
HELPERS_LOADED=1

log() {
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1"
}
```
#### ¿Qué pasa aquí?
  1. `[[ -n "$HELPERS_LOADED" ]]`
    * Verifica si la variable ya existe
  2. Si existe -> `return`
    * Sale del archivo sin ejecutar nada más
  3. Si no existe:
    * Define la variable
    * Carga funciones

#### ¿Por qué usar return y no exit?
  * return → sale del archivo sourced
  * exit → mata todo el script

#### Forma más robusta
```bash
if [[ "${HELPERS_LOADED:-}" == "1" ]]; then
  return
fi
readonly HELPERS_LOADED=1
```
Mejora:
  * Evita errores si la variable no existe
  * `readonly` evita que se modifique accidentalmente

#### Relación
  * Helpers → funciones reutilizables
  * Guard → evita cargar helpers múltiples veces

#### Buenas prácticas
  * Usar nombres de guard únicos `MYAPP_HELPERS_LOADED=1`
  * No pongas lógica ejecutable fuera de funciones (evitar efectos al hacer `source`)
  * Mantén helpers pequeños y enfocados


##  Diferencias

### 1. Helper | Lo más básico
  * Nivel: pequeño
  * Forma típica: función
  * Objetivo: ayudar en tareas concretas

    ```bash
    log() {
        echo "[INFO] $1"
    }
    ```
  Características:
    * Hace **una sola cosa**
    * Es simple
    * Vive dentro de un script o archivo pequeño 

> Esto evita repetir código

### 2. Utility | Herramienta
  * Nivel: medio
  * Forma típica: script ejecutable
  * Objetivo: hacer una tarea útil por sí misma

    ```bash
    #!/bin/bash
    # archivo: backup.sh

    tar -czf backup.tar.gz "$1"
    ```
  Lo ejecutas `./backup.sh carpeta/`

  Características:
    * Puede usarse **independientemente**
    * Hace algo útil completo
    * Puede usar helpers internamente

  > Esto es una herramienta que puedo usar desde la terminal

### 3. Library | Biblioteca
  * Nivel: grande
  * Forma típica: archivo(s) con muchas funciones
  * Objetivo: ser utilizada por otros scripts
    ```bash
    # archivo: string_utils.sh

    to_upper() {
        echo "$1" | tr 'a-z' 'A-Z'
    }

    trim() {
        echo "$1" | xargs
    }
    ```
  Uso:
  ```bash
    source strings_utils.sh

    to_upper "Hola"
  ```

  Características:
    * No se ejecuta sola
    * Se importa con `source`
    * Contiene **muchos helpers relacionados**

> Este archivo es una caja de herramientas reutilizable

## Ejemplo 
Vamos a armar una estructura real de proyecto Bash

### 1. Estructura base
```bash
mi_proyecto/
│
├── bin/
│   └── mi_app.sh        # entrypoint (utility principal)
│
├── lib/
│   ├── helpers.sh       # helpers generales
│   ├── string.sh        # funciones de strings
│   └── file.sh          # funciones de archivos
│
├── config/
│   └── config.sh        # variables/configuración
│
└── README.md
```

### 2. El entrypoint (programa)
Este es el script principal que se ejecuta

`bin/mi_app.sh`
```bash
#!/usr/bin/env bash

# obtener ruta base del proyecto
BASE_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"

# cargar librerías
source "$BASE_DIR/lib/helpers.sh"
source "$BASE_DIR/lib/string.sh"
source "$BASE_DIR/config/config.sh"

main() {
  log "Iniciando aplicación"

  local name="mundo"
  local upper
  upper=$(to_upper "$name")

  log "Hola $upper"
}

main "$@"
```

### 3. Library de Helpers
`lib/helpers.sh`
```bash
# guard de inclusión
if [[ "${HELPERS_LOADED:-}" == "1" ]]; then
  return
fi
readonly HELPERS_LOADED=1

log() {
  echo "[INFO] $1"
}

error() {
  echo "[ERROR] $1" >&2
}
```

### 4. Library específica (strings)
`lib/string.sh`
```bash
# guard de inclusión
if [[ "${STRING_LOADED:-}" == "1" ]]; then
  return
fi
readonly STRING_LOADED=1

to_upper() {
  echo "$1" | tr 'a-z' 'A-Z'
}

to_lower() {
  echo "$1" | tr 'A-Z' 'a-z'
}
```

### 5. Configuración
`config/config.sh`
```bash
# guard
if [[ "${CONFIG_LOADED:-}" == "1" ]]; then
  return
fi
readonly CONFIG_LOADED=1

APP_NAME="MiApp"
VERSION="1.0.0"
```

### 6. Flujo real
Cuando se ejecuta:
```bash
./bin/mi_app.sh
```
Pasa esto:
  * Se calcula `BASE_DIR`
  * Se cargan libreries con `source`
  * Cada archivo usa su **guard** -> evita duplicación
  * Se ejecuta `main`
  * `main` usa helpers (`log`, `to_upper`, etc)